# Modulo de Predicciones - Emergencias

Este notebook construye predicciones operativas interpretables para el sistema distribuido:

- zona con mayor riesgo,
- horario con mayor probabilidad de incidente,
- forecast de incidentes por dia,
- forecast cruzado dia-hora,
- mapa de riesgo.

La salida principal es `predicciones_resumen.json`, consumido por el backend en `/api/predicciones/resumen`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

DATA_PATH = Path('../mdi_detenidosaprehendidos_pm_2026_enero_abril.xlsx')
SHEET = '1. Detenidos y Aprehendidos'
df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
df.shape

## 1. Limpieza y variables temporales/geograficas

In [ ]:
def norm(x):
    if pd.isna(x): return 'SIN_DATO'
    s = str(x).strip()
    return s if s else 'SIN_DATO'

def to_num(x):
    if pd.isna(x): return np.nan
    try: return float(str(x).strip().replace(',', '.'))
    except Exception: return np.nan

def hour_of(x):
    if pd.isna(x): return np.nan
    m = re.search(r'(\d{1,2}):', str(x))
    if not m: return np.nan
    h = int(m.group(1))
    return h if 0 <= h <= 23 else np.nan

work = df.copy()
work['fecha'] = pd.to_datetime(work['fecha_detencion_aprehension'], errors='coerce')
work['hora'] = work['hora_detencion_aprehension'].map(hour_of)
work['lat'] = work['latitud'].map(to_num)
work['lng'] = work['longitud'].map(to_num)
work['fecha_dia'] = work['fecha'].dt.date
work['dia_semana'] = work['fecha'].dt.day_name()
work[['fecha','hora','lat','lng','dia_semana']].head()

## 2. Prediccion de horario con mayor probabilidad

Se calcula la distribucion historica por hora. La hora con mayor probabilidad es la hora con mayor proporcion de registros.

In [ ]:
valid = work[work['fecha'].notna()].copy()
hour_counts = valid['hora'].dropna().astype(int).value_counts().reindex(range(24), fill_value=0)
hour_probs = (hour_counts / hour_counts.sum()).reset_index()
hour_probs.columns = ['hora', 'probabilidad']
hour_probs['label'] = hour_probs['hora'].map(lambda h: f'{h:02d}:00')
hour_probs.sort_values('probabilidad', ascending=False).head(8)

## 3. Prediccion de dia con mayor probabilidad

In [ ]:
orden_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = valid['dia_semana'].value_counts().reindex(orden_dias, fill_value=0)
day_probs = (day_counts / day_counts.sum()).reset_index()
day_probs.columns = ['dia', 'probabilidad']
day_probs.sort_values('probabilidad', ascending=False)

## 4. Forecast de incidentes por dia

Baseline interpretable: promedio historico por dia de semana, ajustado por tendencia simple entre primera y segunda mitad de la serie.

In [ ]:
daily = valid.groupby('fecha_dia').size().sort_index()
weekday_avg = valid.groupby('dia_semana').size() / valid.groupby('dia_semana')['fecha_dia'].nunique()
weekday_avg = weekday_avg.fillna(daily.mean())

ordered = daily.reset_index(name='conteo')
mid = len(ordered) // 2
first = ordered.iloc[:mid]['conteo'].mean()
second = ordered.iloc[mid:]['conteo'].mean()
trend_factor = 1 + max(min(((second - first) / (first or 1)) * 0.25, 0.12), -0.12)

last_date = pd.to_datetime(max(daily.index))
forecast = []
for i in range(1, 15):
    date = last_date + pd.Timedelta(days=i)
    dia = date.day_name()
    pred = max(0, float(weekday_avg.get(dia, daily.mean())) * trend_factor)
    forecast.append({'fecha': date.date().isoformat(), 'dia': dia, 'incidentes_predichos': round(pred)})
pd.DataFrame(forecast)

## 5. Prediccion de zona con mayor riesgo

Score interpretable por distrito:

`score = total + casos_con_arma * 2.5 + casos_nocturnos * 0.75`

Esto prioriza volumen historico, armas y eventos nocturnos.

In [ ]:
sin_arma = {'NINGUNA','SIN_DATO','SE_DESCONOCE'}
work['tiene_arma'] = ~work['arma'].map(norm).str.upper().isin(sin_arma)
work['nocturno'] = work['hora'].fillna(-1).between(20,23) | work['hora'].fillna(-1).between(0,5)

risk = work.groupby(['nombre_provincia','nombre_canton','nombre_distrito']).agg(
    total=('tipo','size'),
    con_arma=('tiene_arma','sum'),
    nocturnos=('nocturno','sum')
).reset_index()
risk['score'] = risk['total'] + risk['con_arma'] * 2.5 + risk['nocturnos'] * 0.75
risk['probabilidad_relativa'] = risk['score'] / risk['score'].max()
risk.sort_values('score', ascending=False).head(15)

## 6. Forecast dia-hora

Cruza el forecast diario con la probabilidad historica por hora para identificar ventanas de atencion.

In [ ]:
top_hours = hour_probs.sort_values('probabilidad', ascending=False).head(5)
slots = []
for row in forecast[:7]:
    for _, h in top_hours.iterrows():
        slots.append({
            'fecha': row['fecha'],
            'dia': row['dia'],
            'hora': h['label'],
            'incidentes_predichos': round(row['incidentes_predichos'] * h['probabilidad'], 2)
        })
pd.DataFrame(slots).sort_values('incidentes_predichos', ascending=False).head(20)

## 7. Exportacion JSON para el backend

In [ ]:
# En la app ya se genera un JSON equivalente en backend/src/datos/predicciones.
# Este bloque muestra la estructura esperada.
resumen = {
    'metadata': {'modelo': 'baseline_estadistico_interpretable', 'filas': int(len(work))},
    'kpis': {
        'zonaMayorRiesgo': risk.sort_values('score', ascending=False).iloc[0]['nombre_distrito'],
        'horaMayorProbabilidad': hour_probs.sort_values('probabilidad', ascending=False).iloc[0]['label'],
        'forecastPromedioDiario': float(np.mean([x['incidentes_predichos'] for x in forecast]))
    }
}
resumen